In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%writefile /content/drive/MyDrive/software-build-failure-prediction/data/README.md
# Dataset Documentation

## Source
TravisTorrent dataset (final-2017-01-25.csv.gz), obtained from https://testroots.github.io/travistorrent-site
Citation: Beller, M., Gousios, G., & Zaidman, A. (2017). TravisTorrent: Synthesizing Travis CI and GitHub
for Full-Stack Research on Continuous Integration. MSR 2017.

## Filtering Steps
1. Loaded the raw CSV.gz in chunks (chunksize=50,000) to handle the 2.64M-row original dataset on limited hardware.
2. Filtered to `gh_lang == 'ruby'` (898 of 1,300 projects in TravisTorrent are Ruby).
3. Filtered to `tr_status` in ['passed', 'failed'], dropping 'errored' and 'canceled' builds (ambiguous outcomes).
4. Aggregated from job-level rows to build-level rows (grouped by `tr_build_id`), since the raw dataset
   contains one row per CI job, with multiple jobs per build.

## Resulting Dataset Sizes
- Raw filtered (job-level): 1,542,248 rows × 66 columns
- Aggregated (build-level): 252,151 rows × 66 columns
- Class distribution: 197,850 passed (78.5%) / 54,301 failed (21.5%)

## Feature Split
- `df_clean.csv`: 27 pre-execution features + target (features available before a build runs)
- `df_leaky.csv`: pre-execution + post-execution features + target (includes test results, build duration —
  used only for the leaky baseline comparison)

See `src/feature_split.py` for the exact column lists.

In [ ]:
%%writefile /content/drive/MyDrive/software-build-failure-prediction/src/__init__.py

Writing /content/drive/MyDrive/software-build-failure-prediction/src/__init__.py


In [ ]:
%%writefile /content/drive/MyDrive/software-build-failure-prediction/src/feature_split.py
"""
Defines the pre-execution vs. post-execution feature split for the
Intelligent Software Build Failure Prediction project.

Pre-execution: available before a CI build runs.
Post-execution ("leaky"): only available after/during the build (test results, duration, etc.)
"""

PRE_EXECUTION_FEATURES = [
    'git_branch', 'git_prev_commit_resolution_status',
    'gh_is_pr', 'gh_pull_req_num', 'gh_lang', 'gh_team_size',
    'gh_num_issue_comments', 'gh_num_commit_comments', 'gh_num_pr_comments',
    'git_diff_src_churn', 'gh_diff_files_added', 'gh_diff_files_deleted',
    'gh_diff_files_modified', 'gh_diff_src_files',
    'gh_diff_doc_files', 'gh_diff_other_files', 'gh_num_commits_on_files_touched', 'gh_sloc',
    'gh_test_lines_per_kloc', 'gh_test_cases_per_kloc', 'gh_asserts_cases_per_kloc',
    'gh_by_core_team_member', 'gh_description_complexity', 'gh_repo_age',
    'gh_repo_num_commits', 'tr_build_number', 'gh_build_started_at'
]

POST_EXECUTION_FEATURES = [
    'tr_log_status', 'tr_log_setup_time', 'tr_log_analyzer', 'tr_log_frameworks',
    'tr_log_bool_tests_ran', 'tr_log_bool_tests_failed', 'tr_log_num_tests_ok',
    'tr_log_num_tests_failed', 'tr_log_num_tests_run', 'tr_log_num_tests_skipped',
    'tr_log_num_test_suites_run', 'tr_log_num_test_suites_ok', 'tr_log_num_test_suites_failed',
    'tr_log_tests_failed', 'tr_log_testduration', 'tr_log_buildduration', 'tr_duration', 'tr_log_lan'
]

DROP_COLUMNS = [
    'tr_build_id', 'gh_project_name', 'git_merged_with', 'tr_virtual_merged_into',
    'tr_job_id', 'tr_original_commit', 'tr_jobs'
]

ZERO_VARIANCE_COLUMNS = [
    'git_diff_test_churn', 'gh_diff_tests_added', 'gh_diff_tests_deleted'
]

Writing /content/drive/MyDrive/software-build-failure-prediction/src/feature_split.py


In [ ]:
%%writefile /content/drive/MyDrive/software-build-failure-prediction/src/data_loader.py
"""
Chunked loading and build-level aggregation of the TravisTorrent dataset.
"""

import pandas as pd


def load_and_filter_raw(csv_path, language='ruby', chunksize=50_000):
    """
    Load the raw TravisTorrent CSV.gz in chunks, filtering to a single language
    and resolved build outcomes (passed/failed) to keep memory usage manageable.
    """
    chunks = []
    for chunk in pd.read_csv(csv_path, compression='gzip', chunksize=chunksize, low_memory=False):
        filtered = chunk[
            (chunk['gh_lang'] == language) &
            (chunk['tr_status'].isin(['passed', 'failed']))
        ]
        chunks.append(filtered)
    return pd.concat(chunks, ignore_index=True)


def aggregate_to_build_level(df):
    """
    Collapse job-level rows (one row per Travis CI job) into build-level rows
    (one row per tr_build_id), since a single build can trigger multiple jobs.
    Pre-execution metadata is identical across jobs of the same build (take 'first'),
    while test-log fields are aggregated (summed / max) across jobs.
    """
    log_agg_cols = [
        'tr_log_num_tests_failed', 'tr_log_num_tests_ok', 'tr_log_num_tests_run',
        'tr_log_bool_tests_failed', 'tr_log_testduration', 'tr_log_buildduration'
    ]
    agg_dict = {col: 'first' for col in df.columns if col not in log_agg_cols}
    agg_dict.update({
        'tr_log_num_tests_failed': 'sum',
        'tr_log_num_tests_ok': 'sum',
        'tr_log_num_tests_run': 'sum',
        'tr_log_bool_tests_failed': 'max',
        'tr_log_testduration': 'sum',
        'tr_log_buildduration': 'sum',
    })
    return df.groupby('tr_build_id', as_index=False).agg(agg_dict)

Writing /content/drive/MyDrive/software-build-failure-prediction/src/data_loader.py


In [ ]:
%%writefile /content/drive/MyDrive/software-build-failure-prediction/src/preprocessing.py
"""
Preprocessing utilities: missing value handling, encoding, and time-based splitting.
"""

import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler


def create_target(df, status_col='tr_status'):
    """Map tr_status ('passed'/'failed') to a binary target column."""
    df = df[df[status_col].isin(['passed', 'failed'])].copy()
    df['target'] = (df[status_col] == 'failed').astype(int)
    return df


def fill_missing_values(df):
    """Fill missing values: 0 for PR-related counts, median for numeric, 'unknown' for categorical."""
    for col in ['gh_pull_req_num', 'gh_num_issue_comments', 'gh_num_pr_comments']:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    numeric_cols = df.select_dtypes(include='number').columns
    for col in numeric_cols:
        df[col] = df[col].fillna(df[col].median())

    categorical_cols = df.select_dtypes(include='object').columns
    for col in categorical_cols:
        df[col] = df[col].fillna('unknown')

    return df


def time_based_split(df, time_col='gh_build_started_at', train_frac=0.8):
    """Sort by build start time and split chronologically to avoid future-information leakage."""
    df = df.sort_values(time_col)
    split_idx = int(len(df) * train_frac)
    return df.iloc[:split_idx], df.iloc[split_idx:]


def encode_and_scale(X_train, X_test):
    """Label-encode categorical columns and standard-scale numeric columns, fit on train only."""
    X_train, X_test = X_train.copy(), X_test.copy()

    categorical_cols = X_train.select_dtypes(include='object').columns.tolist()
    for col in categorical_cols:
        le = LabelEncoder()
        X_train[col] = le.fit_transform(X_train[col].astype(str))
        X_test[col] = X_test[col].astype(str).map(lambda x: x if x in le.classes_ else le.classes_[0])
        X_test[col] = le.transform(X_test[col])

    numeric_cols = X_train.select_dtypes(include='number').columns.tolist()
    scaler = StandardScaler()
    X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
    X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

    return X_train, X_test

Writing /content/drive/MyDrive/software-build-failure-prediction/src/preprocessing.py


In [ ]:
%%writefile /content/drive/MyDrive/software-build-failure-prediction/src/evaluate.py
"""
Model evaluation utilities: metric computation and comparison table generation.
"""

import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score


def evaluate_model(name, pipeline_label, model, X_test, y_test):
    """Compute Precision, Recall, F1, and AUC-ROC for a trained model."""
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    return {
        'Model': name,
        'Pipeline': pipeline_label,
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_proba)
    }


def compute_leakage_drop(results_df, metric='F1'):
    """Compute the % drop in a metric between the Leaky and Clean pipelines, per model."""
    pivot = results_df.pivot(index='Model', columns='Pipeline', values=metric)
    pivot[f'{metric}_drop_%'] = ((pivot['Leaky'] - pivot['Clean']) / pivot['Leaky']) * 100
    return pivot

Writing /content/drive/MyDrive/software-build-failure-prediction/src/evaluate.py


In [ ]:
%%writefile /content/drive/MyDrive/software-build-failure-prediction/requirements.txt
pandas>=2.0.0
numpy>=1.24.0
scikit-learn>=1.3.0
xgboost>=2.0.0
matplotlib>=3.7.0
seaborn>=0.12.0
joblib>=1.3.0

Writing /content/drive/MyDrive/software-build-failure-prediction/requirements.txt


In [ ]:
%%writefile /content/drive/MyDrive/software-build-failure-prediction/README.md
# Intelligent Software Build Failure Prediction Framework

A machine learning project investigating data leakage in CI build failure prediction,
using the TravisTorrent dataset.

## Research Question
Prior work on build failure prediction reports very high accuracy (95%+), but often uses
features only available *after* a build completes (test results, build duration). This
project quantifies the performance drop when models are restricted to features genuinely
available *before* a build runs — a realistic, deployable prediction setting.

## Pipeline
1. `notebooks/01_data_loading_filtering.ipynb` — load and filter TravisTorrent to Ruby, resolved builds
2. `notebooks/02_feature_engineering.ipynb` — split into pre-execution vs. post-execution features
3. `notebooks/03_eda.ipynb` — class balance, distributions, correlations
4. `notebooks/04_modeling_leaky_baseline.ipynb` — train LR/RF/XGBoost on all features
5. `notebooks/05_modeling_clean_pipeline.ipynb` — train LR/RF/XGBoost on pre-execution features only
6. `notebooks/06_comparison_results.ipynb` — compare both pipelines, feature importance, final figures

## Key Result
| Model | F1 (Leaky) | F1 (Clean) | Drop |
|---|---|---|---|
| Logistic Regression | 0.517 | 0.331 | 35.9% |
| Random Forest | 0.799 | 0.181 | 77.3% |
| XGBoost | 0.751 | 0.356 | 52.5% |

All models show substantial performance degradation once post-execution (leaky) features
are removed, confirming that realistic pre-execution build failure prediction is a
meaningfully harder task than prior literature's reported accuracy suggests.

## Dataset
TravisTorrent (Beller et al., 2017) — see `data/README.md` for filtering details.

## Reproducing
Run notebooks 01–06 in order inside Google Colab, with this repository's folder
structure mounted via Google Drive. See `requirements.txt` for dependencies.

Writing /content/drive/MyDrive/software-build-failure-prediction/README.md
